In [ ]:
try:
    from openmdao.utils.notebook_utils import notebook_mode  # noqa: F401
except ImportError:
    !python -m pip install openmdao[notebooks]

# The `set_input_defaults` function


The `set_input_defaults` function in OpenMDAO is used to specify metadata for inputs that are promoted to the same name within a Group. This is necessary when multiple inputs within a Group are promoted to the same name, but their units or initial values differ. If `set_input_defaults` is not used in this scenario, OpenMDAO will raise an error during setup.

The function does not set the actual values of the inputs, only the metadata that will be used to create the [_auto_ivc](../../../other_useful_docs/auto_ivc_api_translation.ipynb)
output connected to them. `set_input_defaults` is typically called during the model setup phase, within the setup method of a Group.


```{eval-rst}
    .. automethod:: openmdao.core.group.Group.set_input_defaults
        :noindex:
```

## How it differs from the `set_val` function

While both `set_input_defaults` and `set_val` deal with variable management in OpenMDAO, they have distinct purposes and are used in different contexts.

- `set_input_defaults` is used at the group level to define default metadata (units and initial value) for promoted inputs, specifically to resolve ambiguity when multiple inputs are promoted to the same name. This is crucial for inputs connected to the automatically generated `_auto_ivc` component.

  - Used to resolve inconsistencies between auto-IVC values.

  - Specifically used at the group level to specify metadata to be assumed when multiple inputs are promoted to the same name. This is required when the promoted inputs have differing units or values.

  - For example, if two inputs, C1.x and C2.x, are both promoted to x but have different units (e.g., 'ft' and 'inch'), calling `set_input_defaults('x', units='m', val=1.0)` on the parent group will resolve the ambiguity.


- `set_val` is used at the problem level to set the actual value of a variable, including inputs, outputs, and independent variables. It can handle unit conversions and set values for specific indices in array variables.

  - Used at the run script level to set the value of an input variable.

  - Can be used to set the value of a variable in a different unit than its declared unit, and OpenMDAO will perform the conversion.

  - Can be used to set specific indices or index ranges of array variables.

In essence, `set_input_defaults` helps OpenMDAO correctly determine the units and initial values of connected inputs during the setup phase, while `set_val` is used to directly manipulate variable values before or during a run.

*Key Differences*

-  *Scope*: 
   `set_input_defaults` is used at the group level to define default metadata for promoted inputs, while `set_val` is used at the problem level to set specific values for variables.

- *Purpose*: 
  `set_input_defaults` resolves ambiguities when multiple inputs are promoted to the same name, while `set_val` is used to assign values to variables.

- *Timing*: 
  `set_input_defaults` is typically called during the model setup phase, while `set_val` can be called before or during a run of the model.

## Connected Inputs Without a Source

If multiple inputs have been promoted to the same name but *not* connected manually to an output or promoted
to the same name as an output, then the framework will connect all of those inputs to an
`_auto_ivc` output.  If, however, there is any difference between the units or values of any of those inputs,
then you must tell the framework what units and/or values to use when creating the corresponding
`_auto_ivc` output.  You do this by calling the `set_input_defaults` function using the promoted
input name on a Group that contains all of the promoted inputs.

```{eval-rst}
    .. automethod:: openmdao.core.group.Group.set_input_defaults
        :noindex:
```

Below is an example of what you'll see if you do *not* call `set_input_defaults` to disambiguate
your units and/or values:

In [1]:
import openmdao.api as om

prob = om.Problem(name='no_set_input_defaults')
prob.model.add_subsystem('C1', om.ExecComp('y=x*2.',
                                             x={'val': 1.0, 'units': 'ft'},
                                             y={'val': 0.0, 'units': 'ft'}),
                         promotes=['x'])
prob.model.add_subsystem('C2', om.ExecComp('y=x*3.',
                                             x={'val': 1.0, 'units': 'inch'},
                                             y={'val': 0.0, 'units': 'inch'}),
                         promotes=['x'])

try:
    prob.setup()
except RuntimeError as err:
    print(str(err))


Collected errors for problem 'no_set_input_defaults':
   <model> <class Group>: The following inputs, ['C1.x', 'C2.x'], promoted to 'x', are connected but their metadata entries ['units', 'val'] differ. Call <group>.set_input_defaults('x', units=?, val=?), where <group> is the model to remove the ambiguity.


In [ ]:
try:
    prob.setup()
except RuntimeError as err:
    print(str(err))
    assert(str(err) == "\nCollected errors for problem 'no_set_input_defaults':\n   <model> <class Group>: The following inputs, ['C1.x', 'C2.x'], promoted to 'x', are connected but their metadata entries ['units', 'val'] differ. Call <group>.set_input_defaults('x', units=?, val=?), where <group> is the model to remove the ambiguity.")
else:
    raise RuntimeError("Exception expected.")

The next example shows a successful run after calling `set_input_defaults`:

In [2]:
import openmdao.api as om

prob = om.Problem()
G1 = prob.model.add_subsystem('G1', om.Group())
G1.add_subsystem('C1', om.ExecComp('y=x*2.',
                                    x={'val': 1.0, 'units': 'cm'},
                                    y={'val': 0.0, 'units': 'cm'}),
                 promotes=['x'])
G1.add_subsystem('C2', om.ExecComp('y=x*3.',
                                    x={'val': 1.0, 'units': 'mm'},
                                    y={'val': 0.0, 'units': 'mm'}),
                 promotes=['x'])

# units and value to use for the _auto_ivc output are ambiguous.  This fixes that.
G1.set_input_defaults('x', units='m', val=1.0)

prob.setup()

# set G1.x to 2.0 m, based on the units we gave in the set_input_defaults call
prob.set_val('G1.x', 2.)

prob.run_model()

# we gave 'G1.x' units of 'm' in the set_input_defaults call
print(prob.get_val('G1.x'))

prom='G1.x' metalist=[{'path': '', 'prom': 'G1.x', 'auto': True}, {'prom': 'x', 'auto': False, 'val': 1.0, 'units': 'm', 'src_shape': (1,), 'path': 'G1'}]
prefix='' prom='G1.x' DISCRETE  prom2abs_in[prom]=['G1.C1.x', 'G1.C2.x']
tgt='G1.C1.x' (tgt in abs_in2prom_info)=False
tgt='G1.C2.x' (tgt in abs_in2prom_info)=False
META ==============
{'auto': True, 'path': '', 'prom': 'G1.x'}
FULLMETA ==============
{'units': 'm', 'val': 1.0}
prefix='G1.' meta['val']=1.0
abs_in='G1.C1.x' 
prefix='G1.' meta['val']=1.0 src_shape=()
META ==============
{'auto': False,
 'path': 'G1',
 'prom': 'x',
 'src_shape': (1,),
 'units': 'm',
 'val': 1.0}
FULLMETA ==============
{'units': 'm', 'val': 1.0}
[2.]


In [ ]:
# using absolute value will give us the value of the input C1.x, in its units of 'cm'
print(prob.get_val('G1.C1.x'))

In [ ]:
# using absolute value will give us the value of the input C2.x, in its units of 'mm'
print(prob.get_val('G1.C2.x'))

In [3]:
from openmdao.utils.assert_utils import assert_near_equal

assert_near_equal(prob.get_val('G1.x'), 2.0, 1e-6)
assert_near_equal(prob.get_val('G1.C1.x'), 200.0, 1e-6)
assert_near_equal(prob.get_val('G1.C2.x'), 2000.0, 1e-6)

0.0

Another possible scenario is to have multiple inputs promoted to the same name when those inputs have
different units, but then connecting them manually to an output using the `connect` function.
In this case, the framework will not raise an exception during setup if `set_input_defaults` was not
called as it does in the case of multiple promoted inputs that connected to `_auto_ivc`.  However,
if the user attempts to set or get the input using the promoted name, the framework *will* raise an
exception if `set_input_defaults` has not been called to disambiguate the units of the promoted
input.  The reason for this difference is that in the unconnected case, the framework won't know
what value and units to assign to the `_auto_ivc` output if they're ambiguous.  In the manually
connected case, the value and units of the output have already been supplied by the user, and
the only time that there's an ambiguity is if the user tries to access the inputs using their
promoted name.

Specifying Units

You can also set an input or request the value of any variable in a different unit than its declared
unit, and OpenMDAO will
perform the conversion for you. This is done with the `Problem` methods `get_val` and `set_val`.

In [4]:
import openmdao.api as om

prob = om.Problem()
prob.model.add_subsystem('comp', om.ExecComp('y=x+1.',
                                             x={'val': 100.0, 'units': 'cm'},
                                             y={'units': 'm'}))

prob.setup()
prob.run_model()

print(prob.get_val('comp.x'))

[100.]


In [ ]:
print(prob.get_val('comp.x', 'm'))

In [ ]:
prob.set_val('comp.x', 10.0, 'mm')
print(prob.get_val('comp.x'))

In [ ]:
print(prob.get_val('comp.x', 'm'))

In [5]:
import openmdao.api as om

prob = om.Problem()
prob.model.add_subsystem('comp', om.ExecComp('y=x+1.',
                                             x={'val': 100.0, 'units': 'cm'},
                                             y={'units': 'm'}))

prob.setup()
prob.run_model()

assert_near_equal(prob.get_val('comp.x'), 100, 1e-6)
assert_near_equal(prob.get_val('comp.x', 'm'), 1.0, 1e-6)
prob.set_val('comp.x', 10.0, 'mm')
assert_near_equal(prob.get_val('comp.x'), 1.0, 1e-6)
assert_near_equal(prob.get_val('comp.x', 'm'), 1.0e-2, 1e-6)

0.0

When dealing with arrays, you can set or get specific indices or index ranges by adding the "indices"
argument to the calls:

In [6]:
import openmdao.api as om
import numpy as np

prob = om.Problem()
prob.model.add_subsystem('comp', om.ExecComp('y=x+1.',
                                             x={'val': np.array([100.0, 33.3]), 'units': 'cm'},
                                             y={'shape': (2, ), 'units': 'm'}))

prob.setup()
prob.run_model()

print(prob.get_val('comp.x'))
print(prob.get_val('comp.x', 'm'))
print(prob.get_val('comp.x', 'km', indices=[0]))

[100.   33.3]
[1.    0.333]
[0.001]


In [ ]:
prob.set_val('comp.x', 10.0, 'mm')
print(prob.get_val('comp.x'))
print(prob.get_val('comp.x', 'm', indices=0))

In [ ]:
prob.set_val('comp.x', 50.0, 'mm', indices=[1])
print(prob.get_val('comp.x'))
print(prob.get_val('comp.x', 'm', indices=1))

In [7]:
import openmdao.api as om
import numpy as np

prob = om.Problem()
prob.model.add_subsystem('comp', om.ExecComp('y=x+1.',
                                             x={'val': np.array([100.0, 33.3]), 'units': 'cm'},
                                             y={'shape': (2, ), 'units': 'm'}))

prob.setup()
prob.run_model()

assert_near_equal(prob.get_val('comp.x'), np.array([100, 33.3]), 1e-6)
assert_near_equal(prob.get_val('comp.x', 'm'), np.array([1.0, 0.333]), 1e-6)
assert_near_equal(prob.get_val('comp.x', 'km', indices=[0]), 0.001, 1e-6)

prob.set_val('comp.x', 10.0, 'mm')
assert_near_equal(prob.get_val('comp.x'), np.array([1.0, 1.0]), 1e-6)
assert_near_equal(prob.get_val('comp.x', 'm', indices=0), 1.0e-2, 1e-6)

prob.set_val('comp.x', 50.0, 'mm', indices=[1])
assert_near_equal(prob.get_val('comp.x'), np.array([1.0, 5.0]), 1e-6)
assert_near_equal(prob.get_val('comp.x', 'm', indices=1), 5.0e-2, 1e-6)

0.0

An alternate method of specifying the indices is by making use of the `slicer` object. This
object serves as a helper function allowing the user to specify the indices value using the same syntax as you would when accessing a numpy array. This example shows that usage.

In [8]:
import openmdao.api as om
import numpy as np

prob = om.Problem()
prob.model.add_subsystem('comp', om.ExecComp('y=x+1.',
                                             x={'val': np.array([[1., 2.], [3., 4.]]), },
                                             y={'shape': (2, 2), }))

prob.setup()
prob.run_model()

print(prob.get_val('comp.x', indices=om.slicer[:, 0]))

[1. 3.]


In [ ]:
print(prob.get_val('comp.x', indices=om.slicer[0, 1]))

In [ ]:
print(prob.get_val('comp.x', indices=om.slicer[1, -1]))

In [ ]:
prob.set_val('comp.x', [5., 6.], indices=om.slicer[:,0])
print(prob.get_val('comp.x', indices=om.slicer[:, 0]))

In [ ]:
prob.run_model()
print(prob.get_val('comp.y', indices=om.slicer[:, 0]))

In [9]:
import openmdao.api as om
import numpy as np

prob = om.Problem()
prob.model.add_subsystem('comp', om.ExecComp('y=x+1.',
                                             x={'val': np.array([[1., 2.], [3., 4.]]), },
                                             y={'shape': (2, 2), }))

prob.setup()
prob.run_model()

assert_near_equal(prob.get_val('comp.x', indices=om.slicer[:, 0]), [1., 3.], 1e-6)
assert_near_equal(prob.get_val('comp.x', indices=om.slicer[0, 1]), 2., 1e-6)
assert_near_equal(prob.get_val('comp.x', indices=om.slicer[1, -1]), 4., 1e-6)

prob.set_val('comp.x', [5., 6.], indices=om.slicer[:,0])
assert_near_equal(prob.get_val('comp.x', indices=om.slicer[:, 0]), [5., 6.], 1e-6)
prob.run_model()
assert_near_equal(prob.get_val('comp.y', indices=om.slicer[:, 0]), [6., 7.], 1e-6)

0.0